In [1]:
import pandas as pd
from IPython.display import display, HTML

data = [
    # 1. 시가총액 & 유동성
    ("시가총액 & 유동성", "일별 시가총액", "심사기준일 기준 시총 순위 산출 - 편입 후보 종목 선별의 핵심 피처", "가능", "KRX 정보데이터시스템 / 한국투자증권·키움 Open API"),
    ("시가총액 & 유동성", "유동 시가총액", "전체 시총이 아닌 유동주식 기준 시총 - KOSPI200 편입 기준에 직접 활용", "가능", "KRX / FnGuide / Bloomberg"),
    ("시가총액 & 유동성", "일평균 거래대금 (3·6개월)", "유동성 기준 충족 여부 판단 - 거래대금 하위 종목 방출 예측", "가능", "KRX / QuantiWise / Dataguide"),
    ("시가총액 & 유동성", "유동비율 (Float Ratio)", "대주주·자사주 제외 유통가능 주식 비율 - 유동시총 계산에 필수", "가능", "한국예탁결제원 / FnGuide"),
    ("시가총액 & 유동성", "거래일수 비율", "관리종목·거래정지 이력 탐지 - 편입 자격 박탈 조건 모델링", "가능", "KRX 공시 / OpenDartReader"),

    # 2. 지수 구성 & 심사 규칙
    ("지수 구성 & 심사 규칙", "현행 KOSPI200 구성 종목 리스트", "현재 편입 종목 파악 -> 방출 후보 역산 / 과거 변경 이력 레이블 생성", "가능", "KRX 지수 공식 홈페이지 (krx.co.kr)"),
    ("지수 구성 & 심사 규칙", "정기 심사 기준일·변경일 이력", "과거 편입/방출 레이블 부여 - 지도학습 타깃 변수 구성", "가능", "KRX 공지사항 / 금융투자협회"),
    ("지수 구성 & 심사 규칙", "KRX 지수 산출 방법론 문서", "Rule-based 피처 설계 근거 (시총 상위 N% 등 공식 기준 반영)", "가능", "KRX 공식 PDF (무료 다운로드)"),
    ("지수 구성 & 심사 규칙", "섹터/산업군 분류 (GICS·KRX)", "섹터 대표성 요건 반영 - 동일 섹터 내 경쟁 종목 간 비교 피처", "가능", "KRX / FnGuide / MSCI GICS"),

    # 3. 재무 데이터
    ("재무 데이터", "분기·연간 매출액 / 영업이익", "수익성 추세 - 편입 후 지수 대표성 유지 여부 예측", "가능", "DART(금융감독원) / OpenDartReader"),
    ("재무 데이터", "자기자본 / 부채비율", "재무 건전성 스코어링 - 관리종목 지정 선행 지표", "가능", "DART / FnGuide / KIS Value"),
    ("재무 데이터", "PER / PBR / EV/EBITDA", "밸류에이션 위치 - 시총 변동 예측의 간접 피처", "가능", "FnGuide / Dataguide / Yahoo Finance"),
    ("재무 데이터", "EPS 컨센서스 변화율", "애널리스트 전망 개선도 -> 주가·시총 상승 선행 신호", "가능", "FnGuide / Wisefn / Bloomberg"),
    ("재무 데이터", "배당수익률 / 배당성향", "기관 선호도 프록시 - 안정적 편입 유지 예측에 보조 활용", "가능", "DART / FnGuide"),

    # 4. 주가 & 기술적 지표
    ("주가 & 기술적 지표", "수정주가 OHLCV (일별)", "시총 순위 계산, 모멘텀/변동성 피처 생성 기반 데이터", "가능", "한국투자증권 API / FinanceDataReader / pykrx"),
    ("주가 & 기술적 지표", "52주 고/저가 대비 위치", "가격 모멘텀 피처 - 신규 상장 후 빠른 시총 상승 패턴 포착", "가능", "pykrx / FinanceDataReader"),
    ("주가 & 기술적 지표", "베타 (vs KOSPI200)", "지수 대표성 측정 - 고베타 종목의 편입 적합성 분석", "가능", "직접 계산 (일별 수익률 데이터)"),
    ("주가 & 기술적 지표", "RSI / 볼린저밴드", "심사 직전 과매수/과매도 국면 파악 - 단기 시총 급변 리스크 피처", "가능", "직접 계산 / TA-Lib"),

    # 5. 소유 구조 & 수급
    ("소유 구조 & 수급", "외국인 지분율 (일별)", "외인 수급 트렌드 - 지수 편입 기대 선매수 패턴 탐지", "가능", "KRX / 한국예탁결제원"),
    ("소유 구조 & 수급", "기관 순매수 누적 (20·60일)", "스마트머니 포지셔닝 - 편입 예상 종목 선행 매집 신호", "가능", "KRX / pykrx"),
    ("소유 구조 & 수급", "공매도 비율 / 대차잔고", "하방 압력 지표 - 방출 후보 종목의 약세 신호 보강", "가능", "KRX 공매도 통계 / 금융투자협회"),
    ("소유 구조 & 수급", "최대주주 지분율 변동", "유동비율 변화 선행 지표 - 대주주 지분 변동이 유동시총에 직접 영향", "가능", "DART 대량보유 공시"),

    # 6. 기업 이벤트 & 공시
    ("기업 이벤트 & 공시", "신규 상장일 / 상장 경과 기간", "상장 후 6개월 요건 충족 시점 예측 - 조기 편입 후보 탐색", "가능", "KRX / DART"),
    ("기업 이벤트 & 공시", "유상증자 / 자사주 소각 공시", "유동주식수 급변 이벤트 -> 유동시총 재산정 트리거 피처", "가능", "DART / OpenDartReader"),
    ("기업 이벤트 & 공시", "합병·분할·지주사 전환 공시", "구조 변경으로 인한 강제 방출/편입 이벤트 사전 탐지", "가능", "DART / KRX 공시"),
    ("기업 이벤트 & 공시", "관리종목·투자주의환기종목 지정", "편입 자격 박탈 조건 직접 피처 - 방출 예측에 높은 중요도", "가능", "KRX / 금융감독원"),
    ("기업 이벤트 & 공시", "감사의견 (적정/한정/거절)", "비적정 의견 -> 상장폐지 위험 -> 방출 확정 선행 지표", "가능", "DART 감사보고서"),

    # 7. 매크로 & 시장 환경
    ("매크로 & 시장 환경", "KOSPI 전체 지수 일별 수익률", "시장 국면 피처 - 강세장/약세장에서 편입 패턴 차이 모델링", "가능", "pykrx / FinanceDataReader"),
    ("매크로 & 시장 환경", "원/달러 환율", "외인 수급 변동 간접 피처 - 환율 약세 시 외인 이탈->시총 하락 연계", "가능", "한국은행 경제통계시스템(ECOS)"),
    ("매크로 & 시장 환경", "국고채 금리 (3년·10년)", "할인율 변화 -> 성장주 밸류에이션 영향 -> 시총 순위 변동 피처", "가능", "한국은행 ECOS / 금융투자협회"),
    ("매크로 & 시장 환경", "글로벌 지수 (S&P500, MSCI EM)", "리스크온/오프 국면 파악 - 외인 수급 방향 예측 보조 피처", "가능", "Yahoo Finance / investing.com"),

    # 8. 대안 데이터
    ("대안 데이터", "뉴스 감성 점수 (종목별)", "긍/부정 뉴스 누적 -> 시총 방향성 선행 신호 (NLP 피처)", "부분가능", "네이버 금융 뉴스 크롤링 / CLOVA Sentiment API"),
    ("대안 데이터", "애널리스트 리포트 수 / 커버리지", "커버리지 확대 = 기관 관심 증가 -> 편입 기대감 상승 신호", "부분가능", "FnGuide / Wisefn (유료) / 증권사 공개 리포트"),
    ("대안 데이터", "ETF 바스켓 예상 편입 리스트", "KODEX200 등 패시브 ETF 리밸런싱 수요 -> 수급 압력 피처", "부분가능", "각 ETF 운용사 공시 / Bloomberg"),
    ("대안 데이터", "포털 검색량 (네이버 DataLab)", "개인 관심도 급증 -> 단기 거래대금 증가 -> 유동성 기준 충족 보조 신호", "가능", "네이버 DataLab API (무료)"),
]

df = pd.DataFrame(data, columns=["카테고리", "필요데이터", "활용아이디어", "수집가능여부", "데이터출처"])

# 카테고리별 색상 매핑
cat_colors = {
    "시가총액 & 유동성":   "#d0e8ff",
    "지수 구성 & 심사 규칙": "#ffe0b2",
    "재무 데이터":          "#f3e5f5",
    "주가 & 기술적 지표":   "#d6f5d6",
    "소유 구조 & 수급":     "#fce4ec",
    "기업 이벤트 & 공시":   "#fff9c4",
    "매크로 & 시장 환경":   "#e0f2f1",
    "대안 데이터":          "#ede7f6",
}

avail_colors = {
    "가능":   ("background-color: #e6f9f0; color: #1a7a4a; font-weight: bold;"),
    "부분가능": ("background-color: #fff3e0; color: #b85c00; font-weight: bold;"),
}

def style_row(row):
    cat_bg = f"background-color: {cat_colors.get(row['카테고리'], '#ffffff')};"
    avail_style = avail_colors.get(row['수집가능여부'], "")
    base = "font-size: 13px; padding: 6px 10px;"
    return [
        cat_bg + base,          # 카테고리
        "font-weight: 600;" + base,  # 필요데이터
        "color: #444;" + base,  # 활용아이디어
        avail_style + base,     # 수집가능여부
        "color: #555;" + base,  # 데이터출처
    ]

styled = (
    df.style
    .apply(style_row, axis=1)
    .set_properties(**{"text-align": "left", "vertical-align": "top", "white-space": "pre-wrap"})
    .set_table_styles([
        {"selector": "thead th",
         "props": [("background-color", "#1e293b"), ("color", "white"),
                   ("font-size", "13px"), ("padding", "10px"), ("text-align", "left")]},
        {"selector": "table",
         "props": [("border-collapse", "collapse"), ("width", "100%")]},
        {"selector": "td, th",
         "props": [("border", "1px solid #dee2e6")]},
        {"selector": "tbody tr:hover td",
         "props": [("filter", "brightness(0.95)")]},
    ])
    .hide(axis="index")
)

pd.set_option("display.max_colwidth", None)
display(styled)
print(f"\n총 {len(df)}개 항목  |  수집가능: {(df['수집가능여부']=='가능').sum()}개  |  부분가능: {(df['수집가능여부']=='부분가능').sum()}개  |  카테고리: {df['카테고리'].nunique()}개")

카테고리,필요데이터,활용아이디어,수집가능여부,데이터출처
시가총액 & 유동성,일별 시가총액,심사기준일 기준 시총 순위 산출 - 편입 후보 종목 선별의 핵심 피처,가능,KRX 정보데이터시스템 / 한국투자증권·키움 Open API
시가총액 & 유동성,유동 시가총액,전체 시총이 아닌 유동주식 기준 시총 - KOSPI200 편입 기준에 직접 활용,가능,KRX / FnGuide / Bloomberg
시가총액 & 유동성,일평균 거래대금 (3·6개월),유동성 기준 충족 여부 판단 - 거래대금 하위 종목 방출 예측,가능,KRX / QuantiWise / Dataguide
시가총액 & 유동성,유동비율 (Float Ratio),대주주·자사주 제외 유통가능 주식 비율 - 유동시총 계산에 필수,가능,한국예탁결제원 / FnGuide
시가총액 & 유동성,거래일수 비율,관리종목·거래정지 이력 탐지 - 편입 자격 박탈 조건 모델링,가능,KRX 공시 / OpenDartReader
지수 구성 & 심사 규칙,현행 KOSPI200 구성 종목 리스트,현재 편입 종목 파악 -> 방출 후보 역산 / 과거 변경 이력 레이블 생성,가능,KRX 지수 공식 홈페이지 (krx.co.kr)
지수 구성 & 심사 규칙,정기 심사 기준일·변경일 이력,과거 편입/방출 레이블 부여 - 지도학습 타깃 변수 구성,가능,KRX 공지사항 / 금융투자협회
지수 구성 & 심사 규칙,KRX 지수 산출 방법론 문서,Rule-based 피처 설계 근거 (시총 상위 N% 등 공식 기준 반영),가능,KRX 공식 PDF (무료 다운로드)
지수 구성 & 심사 규칙,섹터/산업군 분류 (GICS·KRX),섹터 대표성 요건 반영 - 동일 섹터 내 경쟁 종목 간 비교 피처,가능,KRX / FnGuide / MSCI GICS
재무 데이터,분기·연간 매출액 / 영업이익,수익성 추세 - 편입 후 지수 대표성 유지 여부 예측,가능,DART(금융감독원) / OpenDartReader



총 35개 항목  |  수집가능: 32개  |  부분가능: 3개  |  카테고리: 8개
